# Kasus: Prediksi Tingkat Keparahan Kecelakaan Lalu Lintas

Notebook ini menjalankan satu kasus utuh: memuat data, feature engineering, melatih model klasifikasi, mengevaluasinya, menyimpan model dengan **joblib**, dan menjalankan aplikasi **Streamlit**.

Jalankan sel dari atas ke bawah (**Runtime → Run all**). Sel instal dan unggah file hanya berfungsi di Colab.

## 1. Instal library
Sklearn, pandas, dan joblib sudah tersedia di Colab. Streamlit perlu dipasang.

In [ ]:
!pip install -q streamlit

## 2. Muat data
Jika `data/traffic_accident.csv` belum ada, Colab akan meminta Anda mengunggah file CSV.

In [ ]:
import os

DATA_PATH = "data/traffic_accident.csv"
os.makedirs("data", exist_ok=True)

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Pilih file CSV kecelakaan dari komputer Anda...")
    unggahan = files.upload()
    nama_asli = next(iter(unggahan))
    os.replace(nama_asli, DATA_PATH)

print("Data siap:", DATA_PATH)

## 3. Eksplorasi awal
Lihat bentuk data, tipe kolom, missing value, dan distribusi target.

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nMissing value per kolom:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplikat accident_id:", df.duplicated(subset="accident_id").sum())
print("\nDistribusi target:")
print(df["tingkat_keparahan"].value_counts())
df.head()

## 4. Feature engineering: `fitur.py`

Fungsi pembersihan dan feature engineering ditulis ke file `fitur.py`. File ini ikut tersimpan di dalam pipeline, sehingga harus bisa di-import ketika model dimuat lagi.

Dua hal penting di sini:
- `jumlah_korban` **tidak dipakai** sebagai fitur, karena baru diketahui setelah kejadian (data leakage).
- Kolom `hari` asli tidak konsisten dengan `tanggal`, sehingga harinya dihitung ulang dari tanggal.

In [ ]:
%%writefile fitur.py
"""
fitur.py
========
Pembersihan dan feature engineering. Fungsi ini dipakai di dua tempat:
1. saat pelatihan (sebagai langkah pertama pipeline),
2. saat aplikasi Streamlit membuat prediksi.

Karena fungsi ini ikut tersimpan di dalam pipeline (joblib/pickle), fungsi
harus didefinisikan di modul yang bisa di-import (bukan lambda).
"""

import numpy as np
import pandas as pd

# Kolom mentah yang dibutuhkan model (tanpa accident_id, hari, dan jumlah_korban)
KOLOM_MASUKAN = [
    "tanggal", "jam", "kota", "tipe_jalan", "kondisi_cuaca",
    "kondisi_pencahayaan", "kondisi_permukaan_jalan", "jenis_kendaraan",
    "jumlah_kendaraan_terlibat", "batas_kecepatan_kmh", "usia_pengemudi",
    "jenis_kelamin_pengemudi", "alkohol_terdeteksi",
]

# Fitur numerik dan kategorik yang masuk ke model (setelah feature engineering)
NUM = ["jam", "jumlah_kendaraan_terlibat", "batas_kecepatan_kmh", "usia_pengemudi"]
KAT = [
    "kota", "tipe_jalan", "kondisi_cuaca", "kondisi_pencahayaan",
    "kondisi_permukaan_jalan", "jenis_kendaraan", "jenis_kelamin_pengemudi",
    "alkohol_terdeteksi", "hari_dari_tanggal", "akhir_pekan", "musim",
    "kelompok_jam", "kelompok_usia",
]

KELAS = ["Ringan", "Sedang", "Berat"]

NAMA_HARI = {0: "Senin", 1: "Selasa", 2: "Rabu", 3: "Kamis", 4: "Jumat", 5: "Sabtu", 6: "Minggu"}
BULAN_MUSIM_HUJAN = [11, 12, 1, 2, 3, 4]


def fitur_turunan(df):
    """Bersihkan data mentah lalu bentuk fitur turunan. Mengembalikan DataFrame baru."""
    d = df.copy()

    # 1. Normalisasi teks: spasi, simbol '@' yang salah ketik, dan kapitalisasi
    for c in ["kota", "tipe_jalan", "kondisi_pencahayaan", "kondisi_permukaan_jalan",
              "jenis_kendaraan", "jenis_kelamin_pengemudi", "alkohol_terdeteksi"]:
        d[c] = d[c].astype("string").str.strip()
    d["kondisi_cuaca"] = (
        d["kondisi_cuaca"].astype("string").str.strip()
        .str.replace("@", "a", regex=False)     # 'Ber@w@n' -> 'Berawan'
        .str.lower().str.title()                 # 'HUJAN LEBAT' -> 'Hujan Lebat'
    )

    # 2. Fitur dari tanggal (hari dihitung ulang, karena kolom 'hari' asli tidak konsisten)
    tgl = pd.to_datetime(d["tanggal"], errors="coerce")
    d["hari_dari_tanggal"] = tgl.dt.dayofweek.map(NAMA_HARI)
    akhir = np.where(tgl.dt.dayofweek >= 5, "Akhir pekan", "Hari kerja")
    d["akhir_pekan"] = pd.Series(akhir, index=d.index).where(tgl.notna())
    musim = np.where(tgl.dt.month.isin(BULAN_MUSIM_HUJAN), "Musim hujan", "Musim kemarau")
    d["musim"] = pd.Series(musim, index=d.index).where(tgl.notna())

    # 3. Pengelompokan angka menjadi kategori yang bermakna
    d["kelompok_jam"] = pd.cut(
        d["jam"], bins=[0, 6, 11, 15, 18, 24], right=False,
        labels=["Dini hari", "Pagi", "Siang", "Sore", "Malam"],
    ).astype("object")
    d["kelompok_usia"] = pd.cut(
        d["usia_pengemudi"], bins=[0, 25, 40, 55, 200], right=False,
        labels=["<25", "25-39", "40-54", "55+"],
    ).astype("object")

    # 4. Kategori kosong dinyatakan sebagai np.nan agar SimpleImputer aman
    for c in KAT:
        d[c] = d[c].astype("object").where(d[c].notna(), np.nan)

    return d


## 5. Latih dan evaluasi model
Skrip ini membandingkan dua model dengan cross-validation, melatih model terpilih, lalu mengevaluasinya di data test. Hasilnya disimpan di `models/`.

In [ ]:
%%writefile latih_kecelakaan.py
"""
latih_kecelakaan.py
===================
Melatih model klasifikasi tingkat_keparahan (Ringan / Sedang / Berat),
mengevaluasinya, dan menyimpan pipeline ke models/ dengan joblib.
"""

import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, roc_auc_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from fitur import KAT, KELAS, KOLOM_MASUKAN, NUM, fitur_turunan

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# --- 1. Muat data dan buang duplikat ---
df = pd.read_csv("data/traffic_accident.csv")
df = df.drop_duplicates(subset="accident_id").reset_index(drop=True)
df = df.dropna(subset=["tingkat_keparahan"])
print("Data setelah dedup:", df.shape)

X = df[KOLOM_MASUKAN]          # jumlah_korban, accident_id, dan hari TIDAK dipakai
y = df["tingkat_keparahan"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

# --- 2. Preprocessing: imputasi, scaling, dan one-hot ---
prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), NUM),
    ("kat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), KAT),
])

# --- 3. Bandingkan dua model dengan cross-validation di data TRAIN ---
kandidat = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1),
}

skor_cv = {}
for nama, clf in kandidat.items():
    pipe = Pipeline([
        ("fe", FunctionTransformer(fitur_turunan)),
        ("prep", prep),
        ("clf", clf),
    ])
    skor = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1_macro")
    skor_cv[nama] = skor.mean()
    print(f"{nama:20} F1-macro CV = {skor.mean():.3f} (± {skor.std():.3f})")

nama_terpilih = max(skor_cv, key=skor_cv.get)
print("Model terpilih:", nama_terpilih)

# --- 3b. Baseline: tebakan acak sesuai proporsi kelas (pembanding minimum) ---
baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
baseline_pipe = Pipeline([("fe", FunctionTransformer(fitur_turunan)), ("prep", prep), ("clf", baseline)])
baseline_pipe.fit(X_train, y_train)
print(f"Baseline (tebakan acak) F1-macro TEST = "
      f"{f1_score(y_test, baseline_pipe.predict(X_test), average='macro'):.3f}")

# --- 4. Latih model terpilih pada seluruh data train, evaluasi sekali di TEST ---
model = Pipeline([
    ("fe", FunctionTransformer(fitur_turunan)),
    ("prep", prep),
    ("clf", kandidat[nama_terpilih]),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
proba = model.predict_proba(X_test)
kelas_model = list(model.classes_)

metrik = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "f1_macro": f1_score(y_test, y_pred, average="macro"),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
    "roc_auc_ovr_macro": roc_auc_score(
        y_test, proba, multi_class="ovr", average="macro", labels=kelas_model),
}
print("\n=== Metrik pada data TEST ===")
for k, v in metrik.items():
    print(f"{k:20} {v:.3f}")

print("\n=== Classification report ===")
print(classification_report(y_test, y_pred, labels=KELAS, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=KELAS),
    index=[f"Aktual {k}" for k in KELAS],
    columns=[f"Prediksi {k}" for k in KELAS],
)
print("=== Confusion matrix ===")
print(cm)

# --- 5. Simpan bundle dengan joblib ---
Path("models").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)
cm.to_csv("output/confusion_matrix.csv")

fe_df = fitur_turunan(X)
bundle = {
    "pipeline": model,
    "kolom_masukan": KOLOM_MASUKAN,
    "kelas": kelas_model,
    "opsi": {c: sorted(fe_df[c].dropna().unique().tolist())
             for c in ["kota", "tipe_jalan", "kondisi_cuaca", "kondisi_pencahayaan",
                       "kondisi_permukaan_jalan", "jenis_kendaraan",
                       "jenis_kelamin_pengemudi", "alkohol_terdeteksi"]},
    "metrik_test": {k: float(v) for k, v in metrik.items()},
    "confusion_matrix": cm,
    "model_terpilih": nama_terpilih,
    "versi_sklearn": sklearn.__version__,
}
joblib.dump(bundle, "models/keparahan_kecelakaan.joblib", compress=3)
print("\nModel tersimpan di models/keparahan_kecelakaan.joblib")


Jalankan skrip pelatihan. Proses ini memakan waktu beberapa menit karena cross-validation dan Random Forest.

In [ ]:
!python latih_kecelakaan.py

## 6. Muat model dan buat prediksi
Model dimuat dengan joblib. Input cukup berupa data mentah, karena feature engineering sudah ada di dalam pipeline.

In [ ]:
import joblib
import pandas as pd

bundle = joblib.load("models/keparahan_kecelakaan.joblib")
print("Model terpilih:", bundle["model_terpilih"])
print("Metrik test:", {k: round(v, 3) for k, v in bundle["metrik_test"].items()})

kejadian_baru = pd.DataFrame([{
    "tanggal": "2024-06-01", "jam": 22.0, "kota": "Jakarta",
    "tipe_jalan": "Jalan Tol", "kondisi_cuaca": "Hujan Lebat",
    "kondisi_pencahayaan": "Malam", "kondisi_permukaan_jalan": "Licin",
    "jenis_kendaraan": "Mobil", "jumlah_kendaraan_terlibat": 2.0,
    "batas_kecepatan_kmh": 100.0, "usia_pengemudi": 24.0,
    "jenis_kelamin_pengemudi": "Laki-laki", "alkohol_terdeteksi": "Ya",
}])[bundle["kolom_masukan"]]

proba = bundle["pipeline"].predict_proba(kejadian_baru)[0]
for kelas, p in zip(bundle["kelas"], proba):
    print(f"{kelas:8} {p:.1%}")
print("\nPrediksi:", bundle["kelas"][proba.argmax()])

## 7. Confusion matrix dari data test

In [ ]:
bundle["confusion_matrix"]

Unduh file model untuk dipakai di komputer Anda sendiri.

In [ ]:
try:
    from google.colab import files
    files.download("models/keparahan_kecelakaan.joblib")
    files.download("fitur.py")
    files.download("app.py")
except ImportError:
    print("Bukan Colab. File ada di folder models/ dan di folder kerja.")

## 8. Aplikasi Streamlit

`app.py` memuat model dari `models/` dan menampilkan form input. Di Colab, aplikasi berjalan di server lokal Colab, lalu dibuka lewat tunnel publik.

In [ ]:
%%writefile app.py
"""
app.py
======
Aplikasi Streamlit untuk memprediksi tingkat keparahan kecelakaan.
Jalankan: streamlit run app.py
"""

import datetime as dt
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

MODEL_PATH = Path(__file__).parent / "models" / "keparahan_kecelakaan.joblib"


@st.cache_resource
def load_bundle():
    if not MODEL_PATH.exists():
        return None
    return joblib.load(MODEL_PATH)   # memuat pipeline: feature engineering + model


st.set_page_config(page_title="Prediksi Keparahan Kecelakaan", layout="wide")
st.title("Prediksi Tingkat Keparahan Kecelakaan")
st.caption("Model klasifikasi: Ringan, Sedang, atau Berat. Dipakai sebagai alat bantu, bukan keputusan akhir.")

bundle = load_bundle()
if bundle is None:
    st.error("Model belum ada. Jalankan latih_kecelakaan.py terlebih dahulu.")
    st.stop()

opsi = bundle["opsi"]
tab_prediksi, tab_metrik = st.tabs(["Prediksi", "Metrik evaluasi"])

with tab_prediksi:
    kiri, kanan = st.columns(2)
    with kiri:
        tanggal = st.date_input("Tanggal kejadian", value=dt.date(2024, 6, 1),
                                min_value=dt.date(2020, 1, 1), max_value=dt.date(2024, 12, 31))
        jam = st.slider("Jam kejadian", 0, 23, 17)
        kota = st.selectbox("Kota", opsi["kota"])
        tipe_jalan = st.selectbox("Tipe jalan", opsi["tipe_jalan"])
        cuaca = st.selectbox("Kondisi cuaca", opsi["kondisi_cuaca"])
        pencahayaan = st.selectbox("Kondisi pencahayaan", opsi["kondisi_pencahayaan"])
    with kanan:
        permukaan = st.selectbox("Kondisi permukaan jalan", opsi["kondisi_permukaan_jalan"])
        kendaraan = st.selectbox("Jenis kendaraan", opsi["jenis_kendaraan"])
        jumlah_kendaraan = st.number_input("Jumlah kendaraan terlibat", 1, 5, 2)
        batas = st.number_input("Batas kecepatan (km/jam)", 40, 100, 60, step=10)
        usia = st.number_input("Usia pengemudi", 17, 72, 35)
        jk = st.selectbox("Jenis kelamin pengemudi", opsi["jenis_kelamin_pengemudi"])
        alkohol = st.selectbox("Alkohol terdeteksi", opsi["alkohol_terdeteksi"])

    if st.button("Prediksi", type="primary"):
        masukan = pd.DataFrame([{
            "tanggal": str(tanggal),
            "jam": float(jam),
            "kota": kota,
            "tipe_jalan": tipe_jalan,
            "kondisi_cuaca": cuaca,
            "kondisi_pencahayaan": pencahayaan,
            "kondisi_permukaan_jalan": permukaan,
            "jenis_kendaraan": kendaraan,
            "jumlah_kendaraan_terlibat": float(jumlah_kendaraan),
            "batas_kecepatan_kmh": float(batas),
            "usia_pengemudi": float(usia),
            "jenis_kelamin_pengemudi": jk,
            "alkohol_terdeteksi": alkohol,
        }])[bundle["kolom_masukan"]]

        proba = bundle["pipeline"].predict_proba(masukan)[0]
        hasil = pd.Series(proba, index=bundle["kelas"], name="probabilitas")
        pred = hasil.idxmax()

        st.subheader(f"Prediksi: {pred}")
        st.bar_chart(hasil)
        st.dataframe(hasil.map(lambda p: f"{p:.1%}").to_frame())

with tab_metrik:
    st.subheader(f"Model terpilih: {bundle['model_terpilih']}")
    st.markdown("**Metrik pada data test (600 kejadian yang tidak dipakai melatih model):**")
    m = bundle["metrik_test"]
    kolom = st.columns(len(m))
    for col, (nama, nilai) in zip(kolom, m.items()):
        col.metric(nama, f"{nilai:.3f}")
    st.markdown("**Confusion matrix:**")
    st.dataframe(bundle["confusion_matrix"])


Jalankan server Streamlit di latar belakang.

In [ ]:
!streamlit run app.py --server.headless true --server.port 8501 > /content/streamlit.log 2>&1 &

Buka tunnel publik ke port 8501. Salin URL yang muncul di `tunnel.log`.
Password tunnel adalah IP publik Colab, yang ditampilkan oleh perintah berikutnya.

In [ ]:
!npm install -g localtunnel -q
!nohup lt --port 8501 > /content/tunnel.log 2>&1 &
!sleep 5 && cat /content/tunnel.log
!echo "Password tunnel (IP publik):" && curl -s https://loca.lt/mytunnelpassword

Jika aplikasi tidak muncul, cek log Streamlit untuk melihat error.

In [ ]:
!cat /content/streamlit.log